# Diff-SSL TVC-LSTM Baseline — Multi-Setting Direct Output

**Google Colab**: Runtime → **GPU**. Open via *File → Open notebook → GitHub*
(`5aola/Virtual-Analogue-Compressor-Modelling`); cell 1 clones the repo and mounts
Drive for the dataset. **Push local changes before running.**

Ablation **base model**: the conditioned LSTM from the diffssl paper recipe
(`LSTM32TVC` / `LSTM96TVC` — `cond_type="tvcond"`, `TVFiLMCond` + sample-rate
LSTM, `0.5·L1 + 0.5·MR-STFT`, AdamW + ReduceLROnPlateau, TBPTT
`step_num_samples=4410`), built via the published [nablafx](https://github.com/mcomunita/nablafx)
package (`pip install nablafx`; the gitignored `external/` checkout is not in
this repo).

**Dataset / split** — identical to `05_conditioning/train_lstm_tfilm_gr.ipynb`:
- 10 settings × 10 songs (GR-curve inventory gates pairs; wet WAV is the target)
- seed 42: val = 1 song × all settings; test = held-out songs × lowest-threshold settings
- Stateful multi-stream TBPTT (`segment_len=32768` chunks, LSTM state carried chunk→chunk)

**Training budget**: fixed **100 epochs** (no early stopping).

In [ ]:
# ── 0. Dependencies ──────────────────────────────────────────────────
!pip install -q "numpy>=2.0,<2.6"
!pip install -q torchmetrics soundfile auraloss einops lightning-utilities packaging
!pip install -q --no-deps lightning nablafx

import sys, types

rational = types.ModuleType("rational")
rational.torch = types.ModuleType("rational.torch")
rational.torch.Rational = type("Rational", (), {})
sys.modules["rational"], sys.modules["rational.torch"] = rational, rational.torch

# diffssl nablafx.system imports FAD — stub so we never pull tensorflow/encodec.
fad = types.ModuleType("frechet_audio_distance")
fad.FrechetAudioDistance = type("FrechetAudioDistance", (), {})
sys.modules["frechet_audio_distance"] = fad

import numpy as np, torch
assert np.__version__.startswith("2."), f"numpy {np.__version__} — restart runtime, re-run cell 0"
print(f"numpy {np.__version__}, torch {torch.__version__}")

In [ ]:
# ── 1. Mount Drive + clone repo ─────────────────────────────────────
# Model code comes from pip ``nablafx`` (cell 0). This repo only supplies
# dataset/system helpers under 02b_sota_training/ (external/ is gitignored).

import os
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

DRIVE_DATA_ROOT = "/content/drive/Othercomputers/MacBook Air/data/Diff-SSL-G-Comp"
REPO_URL = "https://github.com/5aola/Virtual-Analogue-Compressor-Modelling.git"
REPO_ROOT = "/content/Virtual-Analogue-Compressor-Modelling"
SOTA_DIR = os.path.join(REPO_ROOT, "02b_sota_training")
OUTPUT_DIR = os.path.join(os.path.dirname(DRIVE_DATA_ROOT), "diffssl_tvc_runs")

if os.path.isdir(REPO_ROOT):
    !git -C "{REPO_ROOT}" pull --ff-only
else:
    !git clone --depth 1 "{REPO_URL}" "{REPO_ROOT}"

DATA_ROOT = DRIVE_DATA_ROOT
assert os.path.isdir(os.path.join(DATA_ROOT, "gr_curves")), f"Bad DATA_ROOT: {DATA_ROOT}"
assert os.path.isfile(os.path.join(SOTA_DIR, "dataset.py")), f"Clone failed: {REPO_ROOT}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for p in (REPO_ROOT, SOTA_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"SOTA_DIR   : {SOTA_DIR}")
print(f"DATA_ROOT  : {DATA_ROOT}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")

Mounted at /content/drive
Cloning into '/content/Virtual-Analogue-Compressor-Modelling'...
remote: Enumerating objects: 143, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (130/130), done.
remote: Total 143 (delta 14), reused 84 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (143/143), 75.43 MiB | 16.86 MiB/s, done.
Resolving deltas: 100% (14/14), done.


AssertionError: Missing diffssl fork: /content/Virtual-Analogue-Compressor-Modelling/external/nablafx-for-diffssl-compressor

In [ ]:
# ── 2. Cache dataset to Colab local SSD ──────────────────────────────
# Same pair inventory as train_lstm_tfilm_gr: dry + gr_curves gate settings;
# here we also copy the matching wet WAV per (song, setting).

import shutil
from dataset import discover_diffssl_wet_pairs

LOCAL_DATA_ROOT = "/content/Diff-SSL-G-Comp"

pairs = discover_diffssl_wet_pairs(DATA_ROOT)
settings = sorted({p["setting"] for p in pairs})
songs = sorted({p["song"] for p in pairs})
print(f"Caching {len(songs)} songs × {len(settings)} settings → {LOCAL_DATA_ROOT}")

local_dry = Path(LOCAL_DATA_ROOT) / "processed_normalized"
local_dry.mkdir(parents=True, exist_ok=True)
for song in songs:
    fn = f"{song}_UnmasteredWAV.wav"
    src = Path(DATA_ROOT) / "processed_normalized" / fn
    dst = local_dry / fn
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        shutil.copy2(src, dst)

for setting in settings:
    local_gr = Path(LOCAL_DATA_ROOT) / "gr_curves" / setting
    local_gr.mkdir(parents=True, exist_ok=True)
    local_wet = Path(LOCAL_DATA_ROOT) / "processed_ground_truth" / setting
    local_wet.mkdir(parents=True, exist_ok=True)
    for song in songs:
        gr_fn = f"{song}.pt"
        gr_src = Path(DATA_ROOT) / "gr_curves" / setting / gr_fn
        if gr_src.is_file():
            gr_dst = local_gr / gr_fn
            if not gr_dst.exists() or gr_dst.stat().st_size != gr_src.stat().st_size:
                shutil.copy2(gr_src, gr_dst)
        wet_fn = f"{song}-exported.wav"
        wet_src = Path(DATA_ROOT) / "processed_ground_truth" / setting / wet_fn
        if wet_src.is_file():
            wet_dst = local_wet / wet_fn
            if not wet_dst.exists() or wet_dst.stat().st_size != wet_src.stat().st_size:
                shutil.copy2(wet_src, wet_dst)

DATA_ROOT = LOCAL_DATA_ROOT
print(f"Using local cache: {DATA_ROOT}")

In [ ]:
# ── 3. Imports & hyper-parameters (LSTM32TVC / LSTM96TVC) ──────────

import json
from datetime import datetime

import lightning as pl
from lightning.pytorch.callbacks import LearningRateMonitor, ModelCheckpoint, TQDMProgressBar
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger

from dataset import SAMPLE_RATE, SEGMENT_LEN, MultiSettingWetDataModule, discover_diffssl_wet_pairs
from model import build_diffssl_tvc_lstm
from splits import DIFFSSL_PARAM_RANGES, build_split_manifest
from system import DiffSSLTVCLSTMSystem
from src.dsp import PARAM_ORDER

print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "WARNING: CPU runtime")

SPLIT_SEED = 42
N_VAL_SONGS = 1
N_TEST_SONGS = 2

LR = 1e-3
MAX_EPOCHS = 100
STEP_NUM_SAMPLES = 4410          # diffssl LSTM TBPTT sub-step (0.1 s @ 44.1 kHz)

HIDDEN_SIZE = 32                 # LSTM32TVC; set 96 for LSTM96TVC
NUM_LAYERS = 1
COND_TYPE = "tvcond"
COND_BLOCK_SIZE = 128
COND_NUM_LAYERS = 1
NUM_CONTROLS = 4

RUN_TAG = "diffssl_lstm32_tvc_multisetting"
RESUME_RUN = None

In [ ]:
# ── 4. Preview split (same as TFiLM GR notebook) ─────────────────────

preview = build_split_manifest(
    discover_diffssl_wet_pairs(DATA_ROOT),
    seed=SPLIT_SEED,
    n_val_songs=N_VAL_SONGS,
    n_test_songs=N_TEST_SONGS,
)
print(f"Settings ({len(preview.all_settings)}): {preview.all_settings}")
print(f"Test settings (lowest T): {preview.test_settings}")
print(f"Train songs: {preview.train_songs}")
print(f"Val songs  : {preview.val_songs}")
print(f"Test songs : {preview.test_songs}")
print(
    f"Pairs — train={len(preview.train_pair_keys)} "
    f"val={len(preview.val_pair_keys)} test={len(preview.test_pair_keys)}"
)

In [ ]:
# ── 5. Build diffssl model (LSTM32TVC / LSTM96TVC) ─────────────────

model = build_diffssl_tvc_lstm(
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    num_controls=NUM_CONTROLS,
    cond_block_size=COND_BLOCK_SIZE,
    cond_num_layers=COND_NUM_LAYERS,
)
n_params = sum(p.numel() for p in model.parameters())
print(f"BlackBoxModel + LSTM(tvcond, h={HIDDEN_SIZE}): {n_params:,} params")
print(model.processor)

In [ ]:
# ── 6. Train ─────────────────────────────────────────────────────────

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

assert DATA_ROOT.startswith("/content/"), "Run the cache cell first (cell 2)."

if RESUME_RUN:
    RUN_NAME = RESUME_RUN
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = os.path.join(RUN_DIR, "checkpoints", "last.ckpt")
    print(f"RESUMING: {RUN_NAME}")
else:
    RUN_NAME = f"diffssl_tvc_{datetime.now():%Y%m%d_%H%M%S}_{RUN_TAG}"
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = None
    print(f"NEW run: {RUN_NAME}")

os.makedirs(RUN_DIR, exist_ok=True)
split_path = os.path.join(RUN_DIR, "split_manifest.json")

dm = MultiSettingWetDataModule(
    data_root=DATA_ROOT,
    segment_len=SEGMENT_LEN,
    sample_rate=SAMPLE_RATE,
    split_seed=SPLIT_SEED,
    n_val_songs=N_VAL_SONGS,
    n_test_songs=N_TEST_SONGS,
    split_manifest_path=split_path,
)
dm.setup()

print(f"Train/val/test streams: {dm.train_dataset.B} / {dm.val_dataset.B} / {dm.test_dataset.B}")
print(f"Steps/epoch (train): {len(dm.train_dataset)}")
print(f"Split manifest: {split_path}")

with open(os.path.join(RUN_DIR, "hparams.json"), "w") as f:
    json.dump({
        "approach": "diffssl_direct_output_lstm_tvcond",
        "model_type": "nablafx_diffssl_LSTM_tvcond",
        "model_ref": f"experiments/LSTM{HIDDEN_SIZE}TVC/config.yaml",
        "dataset": "Diff-SSL-G-Comp",
        "setting": "multi (10 settings)",
        "conditioning": "tvcond (TVFiLMCond)",
        "sample_rate": SAMPLE_RATE,
        "segment_len": SEGMENT_LEN,
        "step_num_samples": STEP_NUM_SAMPLES,
        "param_order": PARAM_ORDER,
        "param_ranges": DIFFSSL_PARAM_RANGES,
        "split_seed": SPLIT_SEED,
        "test_settings": dm.split.test_settings,
        "train_songs": dm.split.train_songs,
        "val_songs": dm.split.val_songs,
        "test_songs": dm.split.test_songs,
        "model": {
            "hidden_size": HIDDEN_SIZE,
            "num_layers": NUM_LAYERS,
            "cond_type": COND_TYPE,
            "cond_block_size": COND_BLOCK_SIZE,
            "cond_num_layers": COND_NUM_LAYERS,
            "num_controls": NUM_CONTROLS,
            "num_params": n_params,
        },
        "lr": LR,
        "max_epochs": MAX_EPOCHS,
        "loss": "0.5*L1 + 0.5*MR-STFT",
        "optimizer": "adamw + reducelronplateau(0.5,p20)",
        "training": "stateful_streams + diffssl_tbptt_substeps",
    }, f, indent=2)

system = DiffSSLTVCLSTMSystem(
    model=model,
    lr=LR,
    step_num_samples=STEP_NUM_SAMPLES,
)

ckpt_dir = os.path.join(RUN_DIR, "checkpoints")
callbacks = [
    ModelCheckpoint(
        dirpath=ckpt_dir,
        monitor="loss/val",
        mode="min",
        save_top_k=3,
        save_last=True,
        filename="best-{epoch:03d}-{step}",
        auto_insert_metric_name=False,
    ),
    LearningRateMonitor(logging_interval="epoch"),
    TQDMProgressBar(refresh_rate=10),
]
loggers = [
    TensorBoardLogger(save_dir=RUN_DIR, name="tb", version=""),
    CSVLogger(save_dir=RUN_DIR, name="csv", version=""),
]

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator="gpu",
    devices=1,
    callbacks=callbacks,
    logger=loggers,
    log_every_n_steps=10,
)

trainer.fit(system, dm, ckpt_path=_resume_ckpt)

In [ ]:
# ── 7. Test (optional) ───────────────────────────────────────────────

best_ckpt = callbacks[0].best_model_path or os.path.join(ckpt_dir, "last.ckpt")
print(f"Testing with: {best_ckpt}")
trainer.test(system, dm, ckpt_path=best_ckpt)